[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praghav444/CE486-586-GIS-Tutorials/blob/main/tutorials/demos/Lab22_EarthEngine_demo.ipynb)

# Lab 22 demo: Earth Engine + a geospatial foundation model (instructor only)
**Google Earth Engine** holds petabytes of satellite data and runs the analysis on Google's servers. **geemap** (Qiusheng Wu) puts it on an interactive map.

The demo shows **AlphaEarth Foundations**, Google DeepMind's foundation model: every 10 m pixel on Earth, every year, is summarised as **64 numbers** (an *embedding*) learned from many satellites at once. Nobody labelled anything. We ask two questions:
1. Without any labels, which parts of Carroll Creek look alike? (k-means clusters)
2. Which places look like **this** wetland? (similarity search)

![How AlphaEarth Foundations works](https://raw.githubusercontent.com/praghav444/CE486-586-GIS-Tutorials/main/figures/AlphaEarthFoundations.png)

**How it works** (optional reading):
1. **Input:** years of Sentinel-1 radar, Sentinel-2 and Landsat images of the same place, taken on irregular dates, with clouds and gaps.
2. **Encoder:** looks across **space** (the neighbourhood), **time** (how the place changes through the seasons) and fine **detail**, all at once.
3. **Summary:** each 10 m pixel and year becomes **64 numbers**, a point on a sphere. Places that behave alike over the year get similar numbers.
4. **Training without labels:** the 64 numbers must be able to rebuild what the satellites and other data saw (radar, NDVI, GEDI canopy height). So they keep what matters about the surface.
5. **Use:** land cover, change detection, **similarity search** and mapping where labels are scarce. We try two of these below.

**Needs** an Earth Engine account and a Google Cloud project registered for Earth Engine (free for education): https://code.earthengine.google.com/register

In [ ]:
%pip install -q geemap

In [ ]:
import os
os.environ["USE_FOLIUM"] = "1"           # folium maps; must be set before importing geemap
import ee, geemap
PROJECT = "your-cloud-project-id"         # <-- your Earth Engine Cloud project
ee.Authenticate()                         # opens a Google sign-in the first time
ee.Initialize(project=PROJECT)

In [ ]:
ws = ee.FeatureCollection("USGS/WBD/2017/HUC12").filter(ee.Filter.eq("huc12", "031601120412"))
region = ws.geometry()

dem = ee.Image("USGS/3DEP/10m").clip(region)
dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(region)
      .filterDate("2024-01-01", "2025-01-01").select("label").mode().clip(region))
DW_PAL = ["419bdf", "397d49", "88b053", "7a87c6", "e49635", "dfc35a", "c4281b", "a59b8f", "b39fe1"]

m = geemap.Map(center=[33.33, -87.61], zoom=12)
m.add_basemap("SATELLITE")
m.addLayer(ee.Terrain.hillshade(dem), {}, "3DEP hillshade (10 m)")
m.addLayer(dw, {"min": 0, "max": 8, "palette": DW_PAL}, "Dynamic World land cover 2024 (AI, Sentinel-2)")
m.addLayer(ws.style(color="yellow", fillColor="00000000", width=2), {}, "Carroll Creek")
m.add_legend(title="Dynamic World", builtin_legend="Dynamic_World")
m

In [ ]:
# AlphaEarth Foundations satellite embedding: 64 bands per 10 m pixel
emb = (ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterBounds(region)
       .filterDate("2024-01-01", "2025-01-01").mosaic().clip(region))

# 1) No labels at all: let k-means group pixels that look alike
sample = emb.sample(region=region, scale=10, numPixels=4000, seed=1)
clusters = emb.cluster(ee.Clusterer.wekaKMeans(6).train(sample))

# 2) Similarity search: which pixels look like this forested wetland on the valley floor?
pt = ee.Geometry.Point([-87.6245, 33.3113])
ref = emb.reduceRegion(ee.Reducer.mean(), pt.buffer(30), 10)
ref_img = ee.Image.constant(ref.values(emb.bandNames())).rename(emb.bandNames())
similarity = emb.multiply(ref_img).reduce(ee.Reducer.sum())     # dot product: 1 = identical

m = geemap.Map(center=[33.33, -87.61], zoom=12)
m.add_basemap("SATELLITE")
m.addLayer(emb, {"bands": ["A01", "A16", "A09"], "min": -0.3, "max": 0.3}, "Embedding (3 of 64 bands as RGB)")
m.addLayer(clusters.randomVisualizer(), {}, "6 clusters, no labels", False)
m.addLayer(similarity, {"min": 0.5, "max": 1, "palette": ["000000", "2c7fb8", "a1dab4", "ffffcc"]}, "Looks like the wetland")
m.addLayer(pt, {"color": "red"}, "Reference wetland")
m.addLayer(ws.style(color="yellow", fillColor="00000000", width=2), {}, "Carroll Creek")
m

**Talking points:** the similarity layer lights up the valley floors, with no wetland labels and no DEM: the foundation model has learned what bottomland forest looks like from space over the year. Compare with the U-Net from the student notebook, which needed NWI labels **and** HAND. Same caution as always: check it against what you know, and test it somewhere new.